In [3]:
!python my_code.py \
  --csv data.csv \
  --output_dir ./results \
  --method mmseq \
  --threshold 0.3 \
  --parallel_labels 8 \
  --global_dereplicate \
  --global_threshold 0.9 \
  --min_class_size 50

INFO: All required tools found: ['diamond', 'mmseqs']
INFO: ======================================================================
INFO: FAST-PART VERIFICATION (MEMORY-EFFICIENT, DISK-STREAMING)
INFO: ======================================================================
INFO: Streaming CSV → FASTA: data.csv
INFO: Wrote 333884 sequences to /var/folders/xw/ncyd0rkd4_nbf714jz6gctl80000gn/T/fastpart_verify_72vzsqfs/input.fasta (skipped 0 duplicate IDs, 0 empty-label rows, 0 invalid-ID rows)
INFO: --- Phase 2A: Initializing Global MMseqs2 Dereplication (Linclust) ---
INFO: Running: Global MMseqs2 createdb
INFO: Running: Global MMseqs2 clustering at 90.0% identity
INFO: Running: Global MMseqs2 generating cluster TSV
INFO: Parsing cluster representatives...
INFO: Extracting 279992 non-redundant sequences from 333884 total...
INFO: Global clustering completed: Reduced dataset from 333884 to 279992 sequences.
INFO: Class-size filter (min=50): kept 279862 seqs across 171 labels; dropped 31 mino

In [4]:
import pandas as pd
from Bio import SeqIO

# --- Configuration ---
TRAIN_FASTA = "results/train_organism.fasta" 
TEST_FASTA = "results/test_organism.fasta"
DPO_DATA_CSV = "dpo_data.csv"

DPO_TRAIN_OUT = "dpo_train.csv"
DPO_TEST_OUT = "dpo_test.csv"
TEST_CSV_OUT = "test.csv"
OTHER_TRAIN_OUT = "train.csv"

MATCH_COLUMN = "thermo_pid" 

def get_fasta_ids(fasta_path):
    ids = set()
    for record in SeqIO.parse(fasta_path, "fasta"):
        seq_id = record.id.split('|')[0]
        ids.add(seq_id)
    return ids

def get_dpo_ids(dpo_csv, match_column):
    """Extracts all unique protein IDs from the DPO data."""
    df = pd.read_csv(dpo_csv)
    ids = set(df[match_column].unique())
    partner = "meso_pid" if match_column == "thermo_pid" else "thermo_pid"
    if partner in df.columns:
        ids.update(df[partner].unique())
    return ids

def convert_test_fasta_to_csv(fasta_path, output_csv, exclude_ids=None):
    print(f"Converting {fasta_path} to CSV...")
    records = []
    
    for record in SeqIO.parse(fasta_path, "fasta"):
        parts = record.description.split('|')
        entry_name = parts[0]
        
        if exclude_ids and entry_name in exclude_ids:
            continue
            
        organism = parts[1] if len(parts) > 1 else "Unknown"
        sequence = str(record.seq)
        
        records.append({
            "Entry Name": entry_name,
            "Organism": organism,
            "Sequence": sequence
        })
        
    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    print(f"  -> Saved {output_csv} with {len(df):,} sequences.\n")

def convert_remaining_fasta_to_csv(fasta_path, exclude_ids, output_csv):
    print(f"Converting remaining sequences in {fasta_path} to CSV...")
    records = []
    
    for record in SeqIO.parse(fasta_path, "fasta"):
        parts = record.description.split('|')
        entry_name = parts[0]
        
        if entry_name not in exclude_ids:
            organism = parts[1] if len(parts) > 1 else "Unknown"
            sequence = str(record.seq)
            
            records.append({
                "Entry Name": entry_name,
                "Organism": organism,
                "Sequence": sequence
            })
            
    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    print(f"  -> Saved {output_csv} with {len(df):,} sequences.\n")

def filter_dpo_train_set(dpo_csv, train_ids, output_csv, match_column):
    print(f"Filtering {dpo_csv} against Train IDs...")
    
    df = pd.read_csv(dpo_csv)
    initial_rows = len(df)
    
    train_df = df[df[match_column].isin(train_ids)]
    
    train_df.to_csv(output_csv, index=False)
    print(f"  -> Original DPO pairs: {initial_rows:,}")
    print(f"  -> Retained for Train: {len(train_df):,}")
    print(f"  -> Saved {output_csv}.\n")
    
    return set(train_df[match_column])

def filter_dpo_test_set(dpo_csv, test_ids, output_csv, match_column):
    print(f"Filtering {dpo_csv} against Test IDs...")
    
    df = pd.read_csv(dpo_csv)
    initial_rows = len(df)
    
    test_df = df[df[match_column].isin(test_ids)]
    
    test_df.to_csv(output_csv, index=False)
    print(f"  -> Original DPO pairs: {initial_rows:,}")
    print(f"  -> Retained for Test: {len(test_df):,}")
    print(f"  -> Saved {output_csv}.\n")
    
    return set(test_df[match_column])

def main():
    # 1. Extract Test IDs
    print(f"Extracting valid IDs from {TEST_FASTA}...")
    test_ids = get_fasta_ids(TEST_FASTA)
    print(f"  -> Found {len(test_ids):,} unique IDs for testing.\n")
    
    # 2. Get all DPO IDs to exclude from test.csv
    dpo_ids = get_dpo_ids(DPO_DATA_CSV, MATCH_COLUMN)
    
    # 3. Create the paired Test DPO dataset
    dpo_test_ids = filter_dpo_test_set(DPO_DATA_CSV, test_ids, DPO_TEST_OUT, MATCH_COLUMN)
    
    # 4. Convert Test FASTA to Test CSV, excluding any protein in DPO data
    convert_test_fasta_to_csv(TEST_FASTA, TEST_CSV_OUT, exclude_ids=dpo_ids)
    
    # 5. Extract safe Training IDs
    print(f"Extracting valid IDs from {TRAIN_FASTA}...")
    train_ids = get_fasta_ids(TRAIN_FASTA)
    print(f"  -> Found {len(train_ids):,} unique IDs for training.\n")
    
    # 6. Create the paired Train DPO dataset
    used_train_ids = filter_dpo_train_set(DPO_DATA_CSV, train_ids, DPO_TRAIN_OUT, MATCH_COLUMN)
    
    # 7. Extract remaining sequences from Train FASTA not captured in DPO Train
    convert_remaining_fasta_to_csv(TRAIN_FASTA, used_train_ids, OTHER_TRAIN_OUT)
    
    print("All tasks completed successfully!")

if __name__ == "__main__":
    main()

Extracting valid IDs from results/test_organism.fasta...
  -> Found 27,730 unique IDs for testing.

Filtering dpo_data.csv against Test IDs...
  -> Original DPO pairs: 65,532
  -> Retained for Test: 1,408
  -> Saved dpo_test.csv.

Converting results/test_organism.fasta to CSV...
  -> Saved test.csv with 26,860 sequences.

Extracting valid IDs from results/train_organism.fasta...
  -> Found 252,132 unique IDs for training.

Filtering dpo_data.csv against Train IDs...
  -> Original DPO pairs: 65,532
  -> Retained for Train: 49,399
  -> Saved dpo_train.csv.

Converting remaining sequences in results/train_organism.fasta to CSV...
  -> Saved train.csv with 224,646 sequences.

All tasks completed successfully!
